# 第10回 状態空間モデル②：季節調整と欠測の補間

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：埋めた値や取り出した成分は、どのくらい確かなのか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "fable", "feasts", "fda", "KFAS"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(fable)
library(ggtime)   # 分解とACFの図（autoplot）
library(dplyr)
library(ggplot2)
suppressPackageStartupMessages(library(KFAS))   # 状態空間モデル（読み込み時の案内文を出さない）
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

### A-1. 思い出す：ローカルレベルモデルと、欠測

第9回のローカルレベルモデル（local level model）を思い出します。見えない水準$\mu_t$がランダムウォークで動き、観測$y_t$はそれにノイズが乗ったものでした。$\varepsilon_t$の分散を$\sigma_\varepsilon^2$、$\eta_t$の分散を$\sigma_\eta^2$と書きます。

$$
y_t = \mu_t + \varepsilon_t, \qquad \mu_t = \mu_{t-1} + \eta_t
$$

このモデルから、長さ200の系列を作ります。自分で作るので、本当の水準が分かっています。
81番目から30時点分を隠します。

In [ ]:
set.seed(1)
n1 <- 200
level_true <- cumsum(rnorm(n1, sd = 1))     # 水準（ランダムウォーク）
y1 <- level_true + rnorm(n1, sd = 1)        # 観測 ＝ 水準 ＋ ノイズ
gap1 <- 81:110                              # 隠す時点
y1_obs <- replace(y1, gap1, NA)             # 隠した系列（欠測はNA）
ggplot(data.frame(t = 1:n1, y = y1_obs), aes(t, y)) +
  geom_line() +
  labs(x = "Time t (step)", y = "Observation y_t")

`KFAS`パッケージでは、欠測を`NA`のまま渡せます。先に埋めておく必要はありません。
2つの分散（水準のノイズ`Q`、観測のノイズ`H`）は、第9回と同じく最尤法（`fitSSM()`）で推定します。

第9回と違うところが1つあります。第9回では、最初の事前分布を`a1 = 0, P1 = 100^2`と自分で書きました。今日は書きません。
書かないと、KFASは「最初の水準について何も知らない」という事前分布（散漫な事前分布。分散をかぎりなく大きくしたもの）を使います。
あとで状態を8個に増やすので、1つずつ事前分布を書かずにすむように、今日はこの既定に任せます。

In [ ]:
model1 <- SSModel(y1_obs ~ SSMtrend(1, Q = NA), H = NA)                # NA は「推定する」の印
model1 <- fitSSM(model1, inits = c(0, 0), method = "L-BFGS-B",         # 最尤法（初期値は分散の対数）
                 lower = -20, upper = 20)$model
c(Q = model1$Q[1], H = model1$H[1])                                    # 作ったときの値は、どちらも1
out1 <- KFS(model1)                                                    # カルマンフィルタと平滑化

フィルタ（その時点までの観測だけを使う）と、平滑化（すべての観測を使う）を、並べて描きます。
帯は95%の幅、赤い線は本当の水準です。

In [ ]:
est1 <- rbind(
  data.frame(t = 1:n1, kind = "1 filter (uses y up to time t)",
             fit = as.numeric(out1$att), sd = sqrt(as.numeric(out1$Ptt))),
  data.frame(t = 1:n1, kind = "2 smoother (uses all y)",
             fit = as.numeric(out1$alphahat), sd = sqrt(as.numeric(out1$V))))
ggplot(subset(est1, t >= 50 & t <= 140), aes(t)) +
  geom_ribbon(aes(ymin = fit - 1.96 * sd, ymax = fit + 1.96 * sd), fill = "steelblue", alpha = 0.3) +
  geom_line(aes(y = fit), colour = "steelblue") +
  geom_line(data = data.frame(t = 50:140, level = level_true[50:140]), aes(y = level), colour = "red") +
  geom_point(data = data.frame(t = 50:140, y = y1_obs[50:140]), aes(y = y), size = 0.6, na.rm = TRUE) +
  facet_wrap(~kind, ncol = 1) +
  labs(x = "Time t (step)", y = "Level (blue: estimate, red: truth)")

- フィルタ：欠測のあいだは、新しい観測がないので、更新ができません。推定値は最後の値のまま止まり、幅は時点ごとに広がります。
- 平滑化：欠測の右側の観測も使います。推定値は両側を結ぶ直線になり、幅は両端で狭く、まん中で最も広くなります。

幅が正しいかを、本当の水準が帯に入った割合で確かめます。95%の幅なら、およそ95%が入るはずです。

In [ ]:
sm1 <- subset(est1, kind == "2 smoother (uses all y)")
inside <- level_true >= sm1$fit - 1.96 * sm1$sd & level_true <= sm1$fit + 1.96 * sm1$sd
c(all = mean(inside), gap = mean(inside[gap1]),
  sd_at_edge = sm1$sd[min(gap1)], sd_at_middle = sm1$sd[95])

### A-2. 状態を増やす：水準・傾き・季節

第2回と同じ人工の系列を作ります。長さは364日（52週）で、「なめらかな傾向 ＋ 曜日ごとの上下 ＋ ノイズ」です。

In [ ]:
set.seed(1)
n <- 364
day <- 1:n
trend  <- 100 + 8 * sin(2 * pi * day / 364) + 0.02 * day    # なめらかな傾向
season <- rep(c(5, 6, 6, 5, 4, -12, -14), length.out = n)   # 7日でくり返す上下（合計は0）
noise  <- rnorm(n, sd = 2)                                  # ノイズ
sim <- tsibble(day = day, seasonal = trend + season + noise, index = day)
ggplot(sim, aes(day, seasonal)) +
  geom_line() +
  labs(x = "Day", y = "Value (seasonal series)")

状態に、水準$\mu_t$のほかに、傾き$b_t$と季節$s_t$を入れます（第5回のETSの水準$\ell_t$・傾き$b_t$・季節$s_t$にあたります）。

$$
y_t = \mu_t + s_t + \varepsilon_t
$$

- 水準と傾き：$\mu_t = \mu_{t-1} + b_{t-1} + \eta_t$、$b_t = b_{t-1} + \zeta_t$
- 季節：7日分の合計が0。$s_t = -(s_{t-1} + s_{t-2} + \dots + s_{t-6})$

トレンドの型を2つ用意します。

- `trend_order = 1`：水準だけ（傾きなし）。水準がランダムウォークで動く。第9回のローカルレベルモデルに季節を足したもの。
- `trend_order = 2`：水準と傾き。傾きがランダムウォークで動き、水準はそれを足し上げる（$\eta_t$の分散は0）。なめらかなトレンドになる。

In [ ]:
# 状態空間モデル（トレンド ＋ 季節 ＋ ノイズ）を当てはめる。欠測（NA）があってもよい
fit_ssm <- function(y, trend_order = 2, period = 7) {
  model <- if (trend_order == 1) {
    SSModel(y ~ SSMtrend(1, Q = NA) + SSMseasonal(period, Q = 0), H = NA)
  } else {
    SSModel(y ~ SSMtrend(2, Q = list(0, NA)) + SSMseasonal(period, Q = 0), H = NA)
  }
  v <- log(var(diff(y), na.rm = TRUE))                          # 初期値のめやす（分散の対数）
  fitSSM(model, inits = c(v - 3, v), method = "L-BFGS-B",       # 2つの分散を最尤法で推定
         lower = -20, upper = 20)$model                         # 分散の対数を探す範囲
}
# 推定した分散（Q：トレンドを動かすノイズ、H：観測のノイズ）
ssm_var <- function(model) c(Q = max(model$Q), H = model$H[1])

当てはめたモデルから、結果を取り出す関数も用意します。

In [ ]:
# 成分（トレンド、季節）の平滑化による推定値と、95%の幅
ssm_components <- function(model) {
  one <- function(states, name) {
    p <- predict(model, states = states, interval = "confidence", level = 0.95)
    data.frame(t = seq_len(nrow(p)), component = name,
               fit = as.numeric(p[, "fit"]), lwr = as.numeric(p[, "lwr"]), upr = as.numeric(p[, "upr"]))
  }
  rbind(one("trend", "trend"), one("seasonal", "seasonal"))
}
# 観測値の推定値と、95%の幅（欠測の時点では、これが埋めた値とその幅になる）
ssm_fill <- function(model) {
  p <- predict(model, interval = "prediction", level = 0.95)
  data.frame(fit = as.numeric(p[, "fit"]), lwr = as.numeric(p[, "lwr"]), upr = as.numeric(p[, "upr"]))
}
# h時点先までの予測と、95%の幅
ssm_forecast <- function(model, h) {
  p <- predict(model, n.ahead = h, interval = "prediction", level = 0.95)
  data.frame(fit = as.numeric(p[, "fit"]), lwr = as.numeric(p[, "lwr"]), upr = as.numeric(p[, "upr"]))
}
# 誤差（RMSE、MAE）と、本当の値が幅に入った割合
rmse <- function(estimate, truth) sqrt(mean((estimate - truth)^2))
mae  <- function(estimate, truth) mean(abs(estimate - truth))
coverage <- function(p, truth) mean(truth >= p$lwr & truth <= p$upr)

当てはめます。状態の中身（名前）も確かめます。

In [ ]:
model <- fit_ssm(sim$seasonal, trend_order = 2)
ssm_var(model)                        # 作ったときの観測ノイズの分散は 2^2 = 4
colnames(KFS(model)$alphahat)         # 状態ベクトルの中身

状態は8個です。水準（`level`）、傾き（`slope`）、季節が6個（`sea_dummy1`〜`6`。今日と、1日前から5日前までの季節成分）。
同じカルマンフィルタと平滑化が、8個をまとめて推定します。

成分を、95%の幅つきで描きます。赤い破線は、系列を作るときに使った本当の成分です。

In [ ]:
# 成分の図（truth は、本当の成分の表。分からないときは NULL）
plot_components <- function(comp, truth = NULL, x = NULL, x_lab = "Day") {
  if (!is.null(x)) comp$t <- x[comp$t]
  comp$component <- factor(comp$component, levels = c("trend", "seasonal"))
  g <- ggplot(comp, aes(t)) +
    geom_ribbon(aes(ymin = lwr, ymax = upr), fill = "steelblue", alpha = 0.3) +
    geom_line(aes(y = fit), colour = "steelblue") +
    facet_wrap(~component, ncol = 1, scales = "free_y") +
    labs(x = x_lab, y = "Component (band: 95% interval)")
  if (!is.null(truth)) {
    truth$component <- factor(truth$component, levels = c("trend", "seasonal"))
    g <- g + geom_line(data = truth, aes(y = value), colour = "red", linetype = "dashed")
  }
  g
}

In [ ]:
comp <- ssm_components(model)
truth <- rbind(data.frame(t = day, component = "trend", value = trend),
               data.frame(t = day, component = "seasonal", value = season))
plot_components(comp, truth)

数で確かめます。第2回のSTL分解とも比べます。

In [ ]:
tr <- subset(comp, component == "trend")
se <- subset(comp, component == "seasonal")
dcmp <- sim |>
  model(STL(seasonal ~ season(period = 7, window = "periodic"))) |>
  components()
# 季節成分：最初の7日について、推定値、95%の幅、本当の値
data.frame(day = 1:7, estimated = round(se$fit[1:7], 2), lwr = round(se$lwr[1:7], 2),
           upr = round(se$upr[1:7], 2), true = season[1:7])
# トレンド：本当の値との差（RMSE）と、本当の値が幅に入った割合
c(trend_rmse_ssm = rmse(tr$fit, trend), trend_rmse_stl = rmse(dcmp$trend, trend),
  trend_coverage = coverage(tr, trend), trend_width = mean(tr$upr - tr$lwr))

- 季節成分は、本当の値に近く、本当の値は95%の幅の中か、そのすぐ近くにあります。
- トレンドの誤差は、この系列ではSTL分解より小さくなりました。
- いちばんの違いは、成分に幅が付くことです。本当のトレンドは、帯の中にあります。

モデルが系列に合っているかは、第9回と同じく、標準化した予測の外れ（1時点先の予測の外れを、その標準偏差で割ったもの）で確かめます。
モデルが正しければ、iidの標準正規分布に従うはずです。第1回のACFで調べます。

In [ ]:
# 標準化した予測の外れ（1時点先の予測の外れ ÷ その標準偏差）。最初の数時点は、散漫な事前分布のために値が出ないので除く
std_resid <- function(model) {
  e <- as.numeric(rstandard(KFS(model), type = "recursive"))
  data.frame(t = seq_along(e), e = e)[!is.na(e), ]
}

In [ ]:
res <- std_resid(model)
c(mean = mean(res$e), sd = sd(res$e))
res |>
  as_tsibble(index = t) |>
  ACF(e, lag_max = 28) |>
  autoplot() +
  labs(x = "Lag h (day)", y = "Sample ACF of standardized errors")

### A-3. 欠測を、幅つきで埋める

第2回の4つの補間（線形、スプライン、関数データ解析、季節成分を考慮）を持ってきます。計算は第2回と同じです（関数データ解析の関数だけ、短くまとめ直しました）。

In [ ]:
# 線形補間
fill_linear <- function(y) {
  t <- seq_along(y); ok <- !is.na(y)
  approx(t[ok], y[ok], xout = t, rule = 2)$y
}
# スプライン補間（観測点をすべて通る3次スプライン）
fill_spline <- function(y) {
  t <- seq_along(y); ok <- !is.na(y)
  spline(t[ok], y[ok], xout = t, method = "natural")$y
}

In [ ]:
# 関数データ解析：基底関数（Bスプライン、1週間に約1個）と粗さの罰則で、なめらかな関数を推定する
fda_fit <- function(y, lambda) {
  t <- seq_along(y); ok <- !is.na(y)
  basis <- fda::create.bspline.basis(range(t), nbasis = round(length(y) / 7) + 2)
  fda::smooth.basis(t[ok], y[ok], fda::fdPar(basis, 2, lambda))
}
# 関数データ解析による補間（欠測の日だけを、推定した関数の値で埋める）
fill_fda <- function(y, lambda) {
  curve <- as.numeric(fda::eval.fd(seq_along(y), fda_fit(y, lambda)$fd))
  ifelse(is.na(y), curve, y)
}
# GCVが最小になるλを選ぶ
choose_lambda <- function(y, log10_lambda = seq(0, 8, 0.5)) {
  gcv <- sapply(log10_lambda, function(l) sum(fda_fit(y, 10^l)$gcv))
  10^log10_lambda[which.min(gcv)]
}

In [ ]:
# 季節成分を考慮した補間（線形補間で仮に埋める → STL分解 → 季節を引いて線形補間 → 季節を戻す）
fill_seasonal <- function(y, period = 7) {
  y_first <- fill_linear(y)
  s <- stl(ts(y_first, frequency = period), s.window = "periodic")
  s <- as.numeric(s$time.series[, "seasonal"])
  fill_linear(y - s) + s
}

5つ目の方法が、状態空間モデルです。欠測のある系列にそのまま当てはめ、平滑化で埋めます。
ほかの4つと違い、埋めた値に95%の幅が付きます。

In [ ]:
# 5つの方法で埋める。状態空間モデルは、幅（band）も返す
fill_all <- function(y_obs, trend_order = 2, lambda = choose_lambda(y_obs)) {
  band <- ssm_fill(fit_ssm(y_obs, trend_order))
  list(fills = list(linear   = fill_linear(y_obs),
                    spline   = fill_spline(y_obs),
                    fda      = fill_fda(y_obs, lambda),
                    seasonal = fill_seasonal(y_obs),
                    ssm      = band$fit),
       band = band)
}

図を描く関数です。隠した値を黒い点で、状態空間モデルの95%の幅を帯で示します。

In [ ]:
# 隠した期間のまわりを描く（band を渡すと、95%の幅を帯で描く）
plot_fills <- function(y_true, fills, gap, band = NULL, margin = 21) {
  near <- max(1, min(gap) - margin):min(length(y_true), max(gap) + margin)
  long <- do.call(rbind, lapply(names(fills), function(m)
    data.frame(day = gap, method = m, value = fills[[m]][gap])))
  long$method <- factor(long$method, levels = names(fills))
  g <- ggplot()
  if (!is.null(band)) {
    g <- g + geom_ribbon(data = data.frame(day = gap, lwr = band$lwr[gap], upr = band$upr[gap]),
                         aes(day, ymin = lwr, ymax = upr), fill = "steelblue", alpha = 0.3)
  }
  if (length(gap) > 1) {                # 隠したのが1日だけのときは、線を引かない（点だけ）
    g <- g + geom_line(data = long, aes(day, value, colour = method), linewidth = 0.8)
  }
  g +
    geom_line(data = data.frame(day = near, value = replace(y_true, gap, NA)[near]),
              aes(day, value), colour = "grey50", na.rm = TRUE) +
    geom_point(data = data.frame(day = gap, value = y_true[gap]), aes(day, value)) +
    geom_point(data = long, aes(day, value, colour = method), size = 1) +
    labs(x = "Day", y = "Value", colour = "Method")
}

第2回と同じく、150日目から14日間を隠します。まず、状態空間モデルだけを描きます。

In [ ]:
gap <- 150:163                          # 隠す日
y_true <- sim$seasonal
y_obs  <- replace(y_true, gap, NA)      # 隠した系列（欠測はNA）
filled <- fill_all(y_obs)
plot_fills(y_true, filled$fills["ssm"], gap, filled$band)

5つの方法を、RMSEで比べます。状態空間モデルについては、隠した値が幅に入った割合も出します。

In [ ]:
plot_fills(y_true, filled$fills, gap, filled$band)
round(sapply(filled$fills, function(f) rmse(f[gap], y_true[gap])), 2)
c(coverage = coverage(filled$band[gap, ], y_true[gap]),
  width = mean(filled$band$upr[gap] - filled$band$lwr[gap]))

- 誤差は、第2回の「季節成分を考慮した補間」とほぼ同じです。
- 状態空間モデルだけが、幅を付けます。

1か所だけでは、たまたまかもしれません。隠す場所（16か所）と長さ（1日〜21日）を変えて、くり返します。

In [ ]:
# 隠す場所と長さを変えて、方法ごとの誤差の平均と、状態空間モデルの幅に入った割合を出す
gap_study <- function(y, lambda, trend_order = 2, lens = c(1, 3, 7, 14, 21),
                      starts = seq(20, 320, by = 20)) {
  do.call(rbind, lapply(lens, function(len) {
    one <- lapply(starts, function(s) {
      g <- s:(s + len - 1)                                  # 隠す日
      f <- fill_all(replace(y, g, NA), trend_order, lambda)
      list(err = sapply(f$fills, function(v) rmse(v[g], y[g])),
           inside = y[g] >= f$band$lwr[g] & y[g] <= f$band$upr[g])
    })
    err <- rowMeans(sapply(one, function(o) o$err))
    data.frame(len = len, as.list(round(err, 2)),
               ssm_coverage = round(mean(unlist(lapply(one, function(o) o$inside))), 3))
  }))
}

In [ ]:
res_gap <- gap_study(sim$seasonal, lambda = choose_lambda(sim$seasonal))
res_gap

- 状態空間モデルの誤差は、季節成分を考慮した補間と同じくらいで、ノイズの標準偏差（2）に近い値です。
- 隠した値が95%の幅に入った割合は、95%に近くなります。モデルが系列に合っていれば、幅は信用できます。

### A-4. 予測は、右端の欠測である

第2回で、予測は「系列の右端の外にある欠測」だと見ました。状態空間モデルでは、文字どおりそうなります。
右端より先には観測がないので、更新を飛ばして、予測の計算だけをくり返します（第9回の最後に見たとおりです）。

最後の28日を隠して予測し、第2回の3つのベンチマークと比べます。

In [ ]:
h <- 28
train <- sim |> filter(day <= n - h)
test_y <- sim$seasonal[(n - h + 1):n]
fc_ssm <- ssm_forecast(fit_ssm(train$seasonal, trend_order = 2), h)
ggplot(cbind(day = (n - h + 1):n, fc_ssm), aes(day)) +
  geom_ribbon(aes(ymin = lwr, ymax = upr), fill = "steelblue", alpha = 0.3) +
  geom_line(aes(y = fit), colour = "steelblue") +
  geom_line(data = sim |> filter(day > n - 3 * h), aes(y = seasonal)) +
  labs(x = "Day", y = "Value (black: data, blue: forecast with 95% interval)")

In [ ]:
# ベンチマークの誤差（fable）に、状態空間モデルの誤差と、幅に入った割合を足して、1つの表にする
acc_table <- function(fc_bench, data, fc_ssm, test_y) {
  bench <- accuracy(fc_bench, data) |> select(.model, RMSE, MAE)
  bind_rows(bench, tibble(.model = "ssm", RMSE = rmse(fc_ssm$fit, test_y), MAE = mae(fc_ssm$fit, test_y),
                          coverage = coverage(fc_ssm, test_y)))
}

In [ ]:
fc_bench <- train |>
  model(mean   = MEAN(seasonal),
        naive  = NAIVE(seasonal),
        snaive = SNAIVE(seasonal ~ lag(7))) |>
  forecast(h = h)
acc_table(fc_bench, sim, fc_ssm, test_y)

### A-5. 係数を状態にする：時変係数ARモデル

状態に入れられるのは、水準や季節だけではありません。モデルの係数そのものを状態にして、時間とともに動かせます。

AR(1)の係数$\phi$が、時間とともに0.9から$-0.5$まで下がっていく系列を作ります。

$$
y_t = \phi_t\, y_{t-1} + w_t
$$

In [ ]:
set.seed(1)
n2 <- 600
phi_true <- 0.2 + 0.7 * cos(pi * (1:n2) / n2)      # 0.9 から -0.5 まで、なめらかに下がる
w <- rnorm(n2)
x <- numeric(n2)
for (t in 2:n2) x[t] <- phi_true[t] * x[t - 1] + w[t]
ggplot(data.frame(t = 1:n2, x = x), aes(t, x)) +
  geom_line() +
  labs(x = "Time t (step)", y = "Value x_t")

前半はゆっくり上下し、後半は細かく上下します。となりの時点との相関が、時間とともに変わっています。
第1回の言葉でいえば、この系列は定常ではありません。系列を3つに区切って、ラグ1の標本自己相関を出します。

In [ ]:
part <- rep(c("1 first third", "2 middle third", "3 last third"), each = n2 / 3)
tapply(x, part, function(v) acf(v, plot = FALSE)$acf[2])

係数$\phi_t$を状態にします。状態はランダムウォークで少しずつ動く、とします。

- 観測方程式：$y_t = \phi_t\, y_{t-1} + w_t$（状態$\phi_t$に、1時点前の値$y_{t-1}$をかける）
- 状態方程式：$\phi_t = \phi_{t-1} + \eta_t$

In [ ]:
tv <- data.frame(y = x[-1], y_lag = x[-n2])                            # y_t と y_{t-1} を並べる
model_tv <- SSModel(y ~ -1 + SSMregression(~ y_lag, Q = NA), H = NA, data = tv)
model_tv <- fitSSM(model_tv, inits = c(-5, 0), method = "L-BFGS-B", lower = -20, upper = 20)$model
c(Q = model_tv$Q[1], H = model_tv$H[1])
out_tv <- KFS(model_tv)
phi_hat <- data.frame(t = 2:n2, fit = as.numeric(out_tv$alphahat), sd = sqrt(as.numeric(out_tv$V)),
                      truth = phi_true[-1])
phi_fixed <- coef(arima(x, order = c(1, 0, 0), include.mean = FALSE))[["ar1"]]   # 係数が一定のAR(1)
ggplot(phi_hat, aes(t)) +
  geom_ribbon(aes(ymin = fit - 1.96 * sd, ymax = fit + 1.96 * sd), fill = "steelblue", alpha = 0.3) +
  geom_line(aes(y = fit), colour = "steelblue") +
  geom_line(aes(y = truth), colour = "red", linetype = "dashed") +
  geom_hline(yintercept = phi_fixed, colour = "grey40", linetype = "dotted") +
  labs(x = "Time t (step)", y = "AR coefficient phi_t")
c(phi_fixed = phi_fixed,
  coverage = mean(phi_hat$truth >= phi_hat$fit - 1.96 * phi_hat$sd &
                  phi_hat$truth <= phi_hat$fit + 1.96 * phi_hat$sd))

青い線が推定値、帯が95%の幅、赤い破線が本当の係数、点線が「係数が一定のAR(1)」の推定値です。

- 係数が一定のAR(1)は、全体をならした1つの数しか返しません。
- 係数を状態にすると、同じカルマンフィルタと平滑化で、係数の動きを幅つきで追えます。

### A-6. 正規分布でないとき：粒子フィルタ

ここまでの計算（カルマンフィルタと平滑化）は、ノイズが正規分布であることに頼っていました。
正規分布でないときにも使える方法が、粒子フィルタ（particle filter）です。モンテカルロフィルタ（Monte Carlo filter）とも呼ばれます。

考え方は、分布を式（平均と分散）で持つ代わりに、たくさんの粒（その分布から出した乱数）で持つことです。
第9回の「予測 → 観測による更新」を、粒に対して行います。

1. 予測：粒を1つずつ、状態方程式に従って動かす（事前分布の粒になる）
2. 重み：観測に近い粒ほど、重くする（尤度）
3. 選び直す：重みに従って、粒を選び直す（事後分布の粒になる）

欠測の時点では、2と3を飛ばします。カルマンフィルタで、更新を飛ばしたのと同じです。

ローカルレベルモデルの粒子フィルタを、Rの基本の関数だけで書きます。
`move`が状態を動かすノイズ、`weight`が観測のノイズの密度です。この2つを取り替えれば、正規分布でなくても使えます。

In [ ]:
# 粒子フィルタ（水準が1つのモデル）。move：粒を動かす乱数、weight：観測と粒のずれに付ける重み
particle_filter <- function(y, n_particles, move, weight, seed = 1) {
  set.seed(seed)                                    # 乱数を使うので、種を決めておく
  x <- rnorm(n_particles, mean = 0, sd = 5)         # 最初の事前分布 N(0, 5^2) から、粒を作る
  out <- data.frame(t = seq_along(y), mean = NA, lwr = NA, upr = NA)
  for (t in seq_along(y)) {
    x <- x + move(n_particles)                      # 1 予測：粒を1時点ぶん動かす
    if (!is.na(y[t])) {                             #   欠測の時点では、2と3を飛ばす
      w <- weight(y[t] - x)                         # 2 重み：観測に近い粒ほど重くする（尤度）
      x <- sample(x, n_particles, replace = TRUE, prob = w)   # 3 重みに従って、粒を選び直す
    }
    out[t, -1] <- c(mean(x), quantile(x, c(0.025, 0.975)))    # 粒の平均と、95%の幅
  }
  out
}

まず、理論どおりになることを確かめます。A-1の系列（ノイズは正規分布、81番目から30時点が欠測）では、カルマンフィルタが正しい答えを出します。
粒子フィルタのノイズも正規分布にすれば、同じ答えに近づくはずです。2つの分散は、系列を作ったときの値（どちらも1）を与えます。
粒の数を100、1000、10000と変えて、フィルタの平均の差を出します。

In [ ]:
# カルマンフィルタ（KFAS）。粒子フィルタと同じ事前分布にする（最初に1時点動かすので、分散は 5^2 + 1）
# 粒子フィルタの乱数の種は、particle_filter() の中の set.seed(seed) で決めている
kf <- KFS(SSModel(y1_obs ~ SSMtrend(1, Q = 1, a1 = 0, P1 = 5^2 + 1), H = 1))
pf_check <- do.call(rbind, lapply(c(100, 1000, 10000), function(N) {
  pf <- particle_filter(y1_obs, N, move = function(n) rnorm(n, sd = 1), weight = function(e) dnorm(e, sd = 1))
  data.frame(n_particles = N,
             mean_abs_diff = mean(abs(pf$mean - as.numeric(kf$att))),     # 平均の差（200時点の平均）
             max_abs_diff  = max(abs(pf$mean - as.numeric(kf$att))))      # 平均の差（最大）
}))
pf_check

粒を増やすほど、カルマンフィルタとの差は小さくなります。粒子フィルタは、同じベイズ更新を、乱数で近似しています。

次に、正規分布を外します。水準がふだんはほとんど動かず、ときどき大きく跳ぶ系列を作ります。
水準を動かすノイズを、裾の重いコーシー分布（Cauchy distribution）にします。観測のノイズは正規分布（標準偏差1）のままです。

In [ ]:
set.seed(7)
n3 <- 300
level3 <- cumsum(rcauchy(n3, scale = 0.05))     # 水準：コーシー分布のノイズで動く（ときどき大きく跳ぶ）
y3 <- level3 + rnorm(n3, sd = 1)                # 観測 ＝ 水準 ＋ 正規分布のノイズ
ggplot(data.frame(t = 1:n3, y = y3, level = level3), aes(t)) +
  geom_point(aes(y = y), size = 0.6, colour = "grey50") +
  geom_line(aes(y = level), colour = "red") +
  labs(x = "Time t (step)", y = "Value (red: level, dots: observation)")

2つの方法で、水準を推定します。

- カルマンフィルタ：ノイズは正規分布だと考える。2つの分散は、最尤法で推定する。
- 粒子フィルタ：水準のノイズはコーシー分布だと考える。尺度（0.05）と観測のノイズの標準偏差（1）は、系列を作ったときの値を与える（分かっているものとする）。粒は5000個。

In [ ]:
model3 <- SSModel(y3 ~ SSMtrend(1, Q = NA), H = NA)
model3 <- fitSSM(model3, inits = c(0, 0), method = "L-BFGS-B", lower = -20, upper = 20)$model
c(Q = model3$Q[1], H = model3$H[1])
kf3 <- as.numeric(KFS(model3)$att)                                    # カルマンフィルタの平均
pf3 <- particle_filter(y3, 5000, move = function(n) rcauchy(n, scale = 0.05),
                       weight = function(e) dnorm(e, sd = 1))         # 粒子フィルタ
est3 <- rbind(data.frame(t = 1:n3, method = "1 Kalman filter (normal noise)", fit = kf3),
              data.frame(t = 1:n3, method = "2 particle filter (Cauchy noise)", fit = pf3$mean))
ggplot(est3, aes(t)) +
  geom_line(data = data.frame(t = 1:n3, level = level3), aes(y = level), colour = "red") +
  geom_line(aes(y = fit), colour = "steelblue") +
  facet_wrap(~method, ncol = 1) +
  labs(x = "Time t (step)", y = "Level (blue: estimate, red: truth)")
c(rmse_kalman = rmse(kf3, level3), rmse_particle = rmse(pf3$mean, level3))

- カルマンフィルタは、大きな跳びについていくために、水準のノイズの分散を大きく推定します。そのため、跳びのない区間でも、観測のノイズにつられて細かく揺れます。
- 粒子フィルタは、「ふだんは動かないが、まれに大きく跳ぶ」というノイズをそのまま使えます。跳びのない区間では落ち着き、跳んだときにはすぐについていきます。
- ただし、粒子フィルタには、系列を作ったときの尺度を教えています。実データでは、これも推定しなければなりません。
- 粒子フィルタは乱数を使うので、`seed`を変えると、結果が少し変わります。

## B. 実データを解析する

`tsibbledata`パッケージの`vic_elec`を使います。第2回と同じ、オーストラリアのビクトリア州の電力需要です。
第2回のSTL分解・4つの補間・3つのベンチマークと、そのまま比べられます。

B節は、A節で作った関数（`fit_ssm`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
series      <- "Demand"   # 系列："Demand"（1日の電力需要）か "Temperature"（1日の最高気温）
year_from   <- 2014       # 始まりの年：2012〜2014
year_to     <- 2014       # 終わりの年：year_from 以上、2014 以下
trend_order <- 2          # トレンドの型：1（水準だけ。ランダムウォーク）か 2（水準と傾き。なめらか）
gap_from    <- 150        # 隠し始める日（期間の最初の日を1とする通し番号）：30〜300
gap_len     <- 14         # 隠す日数：1〜28
h           <- 28         # 予測する日数：7, 14, 28 のどれか

### B-1. データを確かめる

In [ ]:
glimpse(vic_elec)
interval(vic_elec)
range(vic_elec$Date)
c(rows = nrow(vic_elec), missing = sum(is.na(vic_elec$Demand)), gaps = has_gaps(vic_elec)$.gaps)

- 1行は30分です。`Demand`が電力需要（MWh）、`Temperature`がメルボルンの気温（℃）です。
- 期間は2012年1月1日から2014年12月31日、52608行です。
- 欠測（`NA`）も、行の抜けもありません。

### B-2. 前処理

第2回と同じ前処理です。30分ごとの値を、1日ごとにまとめます。電力需要は1日の合計（GWh）、気温は1日の最高値にします。
そのあと、期間を絞り、設定で選んだ系列を`y`という名前にします。

In [ ]:
daily_all <- vic_elec |>
  index_by(Date) |>
  summarise(Demand = sum(Demand) / 1000,          # 1日の電力需要（GWh）
            Temperature = max(Temperature))       # 1日の最高気温（℃）
interval(daily_all)
daily <- daily_all |>
  filter(format(Date, "%Y") >= year_from, format(Date, "%Y") <= year_to) |>
  mutate(day = row_number(), y = .data[[series]])
y_lab <- if (series == "Demand") "Daily demand (GWh)" else "Daily max temperature (deg C)"
n <- nrow(daily)
n

In [ ]:
ggplot(daily, aes(Date, y)) +
  geom_line() +
  labs(x = "Date", y = y_lab)

### B-3. モデルを当てはめ、成分に分ける

「トレンド ＋ 季節（7日）＋ ノイズ」の状態空間モデルを当てはめます。

In [ ]:
model <- fit_ssm(daily$y, trend_order)
var_hat <- ssm_var(model)
var_hat
comp <- ssm_components(model)
plot_components(comp, x = daily$Date, x_lab = "Date")

第2回のSTL分解のトレンドと重ねます。帯は、状態空間モデルのトレンドの95%の幅です。

In [ ]:
dcmp <- daily |>
  model(STL(y ~ season(period = 7, window = "periodic"))) |>
  components()
tr <- subset(comp, component == "trend")
ggplot(cbind(Date = daily$Date, tr), aes(Date)) +
  geom_line(data = daily, aes(y = y), colour = "grey75") +
  geom_ribbon(aes(ymin = lwr, ymax = upr), fill = "steelblue", alpha = 0.3) +
  geom_line(aes(y = fit), colour = "steelblue") +
  geom_line(data = dcmp, aes(y = trend), colour = "red", linetype = "dashed") +
  labs(x = "Date", y = paste(y_lab, "(blue: state space, red: STL)"))

### B-4. 診断

標準化した予測の外れを調べます。モデルが合っていれば、ACFは破線の中に収まり、分布は標準正規分布に近くなるはずです。

In [ ]:
res <- std_resid(model)
acf_res <- res |>
  as_tsibble(index = t) |>
  ACF(e, lag_max = 28)
autoplot(acf_res) +
  labs(x = "Lag h (day)", y = "Sample ACF of standardized errors")

In [ ]:
ggplot(res, aes(sample = e)) +
  geom_qq(size = 0.8) +
  geom_qq_line(colour = "red") +
  labs(x = "Quantile of standard normal", y = "Standardized error")
resid_check <- c(acf_lag1 = acf_res$acf[1], beyond_2 = mean(abs(res$e) > 1.96), max_abs = max(abs(res$e)))
round(resid_check, 3)

`beyond_2`は、標準化した予測の外れの絶対値が1.96を超えた割合です。正規分布なら、0.05になります。

### B-5. 値を隠して、幅つきで埋める

第2回と同じく、行をわざと抜き、`fill_gaps()`で`NA`の行にします。そのあと、5つの方法で埋めます。

In [ ]:
gap <- gap_from:(gap_from + gap_len - 1)                    # 隠す日
obs <- daily |> filter(!(day %in% gap)) |> fill_gaps()      # 行ごと抜いてから、NAの行を作る
y_true <- daily$y
y_obs  <- obs$y
filled <- fill_all(y_obs, trend_order)
plot_fills(y_true, filled$fills, gap, filled$band) +
  labs(x = "Day", y = y_lab)
gap_rmse <- sapply(filled$fills, function(f) rmse(f[gap], y_true[gap]))
round(gap_rmse, 2)
gap_cover <- c(coverage = coverage(filled$band[gap, ], y_true[gap]),
               width = mean(filled$band$upr[gap] - filled$band$lwr[gap]))
round(gap_cover, 2)

1か所では、幅が正しいかどうかは分かりません。隠す場所を期間の全体に動かし（20日おき、長さは`gap_len`）、隠した値が95%の幅に入った割合を、場所ごとに出します。

In [ ]:
starts <- seq(20, n - gap_len - 10, by = 20)
cover_by_start <- do.call(rbind, lapply(starts, function(s) {
  g <- s:(s + gap_len - 1)
  band <- ssm_fill(fit_ssm(replace(y_true, g, NA), trend_order))
  data.frame(Date = daily$Date[s], start = s, coverage = coverage(band[g, ], y_true[g]),
             width = mean(band$upr[g] - band$lwr[g]), sd_hidden = sd(y_true[g] - band$fit[g]))
}))
cover_all <- mean(cover_by_start$coverage)
ggplot(cover_by_start, aes(Date, coverage)) +
  geom_hline(yintercept = 0.95, linetype = "dashed") +
  geom_col(fill = "steelblue") +
  labs(x = "Start date of the hidden period", y = "Share of hidden values inside the 95% interval")
round(c(coverage_all = cover_all, min = min(cover_by_start$coverage), width = mean(cover_by_start$width)), 3)

### B-6. 予測を比べる

最後の`h`日を隠し、状態空間モデルと3つのベンチマークで予測します。

In [ ]:
train <- daily |> filter(day <= n - h)
test_y <- daily$y[(n - h + 1):n]
fc_ssm <- ssm_forecast(fit_ssm(train$y, trend_order), h)
fc_bench <- train |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y ~ lag(7))) |>
  forecast(h = h)
ggplot(cbind(Date = daily$Date[(n - h + 1):n], fc_ssm), aes(Date)) +
  geom_ribbon(aes(ymin = lwr, ymax = upr), fill = "steelblue", alpha = 0.3) +
  geom_line(aes(y = fit), colour = "steelblue") +
  geom_line(data = daily |> filter(day > n - 3 * h), aes(y = y)) +
  labs(x = "Date", y = paste(y_lab, "(blue: forecast, 95% interval)"))

In [ ]:
acc <- acc_table(fc_bench, daily, fc_ssm, test_y)
acc

### B-7. 時系列交差検証

訓練データを`h`日ずつ伸ばしながら、同じ評価を6回くり返します。

In [ ]:
ends <- n - (6:1) * h                                   # 6回の、訓練データの最後の日
cv_ssm <- do.call(rbind, lapply(ends, function(end) {
  f <- ssm_forecast(fit_ssm(daily$y[1:end], trend_order), h)
  cbind(f, truth = daily$y[(end + 1):(end + h)])
}))
cv_bench <- daily |>
  filter(day <= n - h) |>
  stretch_tsibble(.init = n - 6 * h, .step = h) |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y ~ lag(7))) |>
  forecast(h = h)
acc_cv <- acc_table(cv_bench, daily, cv_ssm, cv_ssm$truth)
acc_cv

読み方：

- B-3：`trend_order = 2`のトレンドは、STL分解のトレンドに近い、なめらかな線になります。ただし、状態空間モデルのほうには幅が付きます。
- B-4：標準化した予測の外れが、ホワイトノイズにも正規分布にもなっていなければ、モデルは系列に合っていません。そのとき、幅は額面どおりには信用できません。
- B-5：幅に入った割合は、季節によって違います。モデルは、ノイズの大きさが1年を通して同じだと仮定しています。電力需要は、そうなっていません。
- B-6、B-7：なめらかなトレンドは、最後の傾きをそのまま先へ延ばします。それが当たるとは限りません。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `series` | `"Demand"` `"Temperature"` |
| `year_from` | 2012〜2014 |
| `year_to` | `year_from`以上、2014以下 |
| `trend_order` | 1（水準だけ）、2（水準と傾き） |
| `gap_from` | 30〜300 |
| `gap_len` | 1〜28 |
| `h` | 7、14、28 |

読むときの観点：

1. トレンドの型（`trend_order`）を1にすると、トレンドの線、診断のACF、予測の誤差は、それぞれどう変わるか。推定された2つの分散（`Q`と`H`）と合わせて読む。
2. 隠す場所（`gap_from`）を夏（30日目のあたり）にすると、隠した値は95%の幅に入るか。B-5の場所ごとの図と合わせて読む。
3. 系列を`"Temperature"`にすると、季節成分の幅は0をまたぐか。曜日ごとの上下がない系列で、季節成分はどう推定されるか。
4. 期間を3年分（`year_from = 2012`）にすると、成分の幅と、幅に入った割合は変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Demand、2014年、`trend_order = 2`、150日目から14日間、28日先）の結果と比べてください。